In [5]:
import sys
import shutil
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path
import kagglehub

In [3]:
load_dotenv()

True

In [4]:
current_dir = Path().cwd()
project_root = current_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("done")

done


In [6]:
data_path = project_root / "data"
cache_path = kagglehub.dataset_download("grouplens/movielens-20m-dataset")
print("Path to dataset files:", cache_path)

for item in Path(cache_path).iterdir():
    shutil.copy2(item, data_path/item.name)
    
print("Data Saved")

100%|██████████| 195M/195M [00:41<00:00, 4.98MB/s] 

Extracting files...


Path to dataset files: C:\Users\BIT\.cache\kagglehub\datasets\grouplens\movielens-20m-dataset\versions\1
Data Saved


In [7]:
for file in data_path.iterdir():
    print(str(file))

c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\Exam_Score_Prediction.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_scores.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\genome_tags.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\link.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\movie.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\rating.csv
c:\Users\BIT\Desktop\Deep_Generative_Modelling\data\tag.csv


In [10]:
movie_df = pd.read_csv(str(data_path / "movie.csv"))
rating_df = pd.read_csv(str(data_path / "rating.csv"))

In [11]:
movie_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [12]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [13]:
movie_df.shape, rating_df.shape

((27278, 3), (20000263, 4))

In [14]:
rating_df = rating_df.iloc[0:1000000]

In [15]:
len(rating_df["userId"].unique())

6743

In [16]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [17]:
rating_df.isna().all()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

In [18]:
movie_df.isna().all()

movieId    False
title      False
genres     False
dtype: bool

In [20]:
# Let's create a pivot table to better understand the data
pivot_ratings_df = rating_df.pivot(index="userId", columns="movieId", values="rating")

In [21]:
pivot_ratings_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,3.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
pivot_ratings_df.shape

(6743, 13950)

In [35]:
pivot_ratings_df_cy = pivot_ratings_df.copy()

In [ ]:
def binaryconvertor(rating_df):
    if rating_df <= 2:
        return 0
    else:
        return 1

In [ ]:
# # pivot_ratings_df.applymap(lambda x: binaryconvertor(x) if pd.notna(x) else x)
# # let's use numpy
# mask = pivot_ratings_df.notna()
# pivot_ratings_df[mask] = pivot_ratings_df[mask].apply(binaryconvertor)

In [38]:
pivot_ratings_df_cy = pd.DataFrame(
    np.where(pivot_ratings_df_cy.isna(), np.nan, (pivot_ratings_df_cy > 2).astype(int)),
    index=pivot_ratings_df_cy.index,
    columns=pivot_ratings_df_cy.columns
)

In [39]:
pivot_ratings_df_cy.head(10)

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1.0,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1.0,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
pivot_ratings_df_cy.fillna(0, inplace=True)

In [42]:
pivot_ratings_df_cy.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,129350,129354,129428,129707,130052,130073,130219,130462,130490,130642
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
pivot_ratings_df_cy.shape

(6743, 13950)

In [44]:
pivot_ratings_df_cy.index

Index([   1,    2,    3,    4,    5,    6,    7,    8,    9,   10,
       ...
       6734, 6735, 6736, 6737, 6738, 6739, 6740, 6741, 6742, 6743],
      dtype='int64', name='userId', length=6743)

In [55]:
pivot_ratings_df.isna().all()

movieId
1         False
2         False
3         False
4         False
5         False
          ...  
130073    False
130219    False
130462    False
130490    False
130642    False
Length: 13950, dtype: bool

Starting to build the RBM

In [57]:
NV = pivot_ratings_df_cy.shape[1]
NH = 100
BATCH_SIZE = 100

In [58]:
weigths = np.random.standard_normal(size=(NV, NH))
a = np.zeros((1, NH))
b = np.zeros((1, NV))

In [59]:
def sigmoid(mat):
    return 1/(1 + (np.exp(-mat)))

In [60]:
# testing it
sample = np.ones((3,4))
sigmoid(sample)

array([[0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858],
       [0.73105858, 0.73105858, 0.73105858, 0.73105858]])

In [ ]:
def sample_h(sample, weigths, biases):
    activation = sample @ weigths + biases
    after_sig = sigmoid(activation)
    